In [1]:
# from sentence_transformers import SentenceTransformer
# import chromadb

# model = SentenceTransformer("all-MiniLM-L6-v2")


# def get_embedding(text: str):
#     return model.encode(text).tolist()

In [2]:
# def create_vector_store():
#     client = chromadb.Client()
#     collection = client.get_or_create_collection(name="research_papers")
#     return collection


# def add_chunks_to_collection(collection, chunks: list[str]):
#     for i, chunk in enumerate(chunks):
#         collection.add(
#             documents=[chunk],
#             embeddings=[get_embedding(chunk)],
#             ids=[f"chunk_{i}"]
#         )

In [3]:
# def retrieve_relevant_chunks(collection, question: str, top_k: int = 3):
#     query_embedding = get_embedding(question)

#     results = collection.query(
#         query_embeddings=[query_embedding],
#         n_results=top_k
#     )

#     documents = results.get("documents", [[]])[0]
#     ids = results.get("ids", [[]])[0]

#     return list(zip(ids, documents))

In [4]:
# question = "What dataset was used in the paper?"
# query_embedding = get_embedding(question)

# results = collection.query(
#     query_embeddings=[query_embedding],
#     n_results=3
# )

# results

In [5]:
# collection = create_vector_store()
# print(collection.count())

In [6]:
# Step 1: Import libraries and load embedding model
from sentence_transformers import SentenceTransformer
import chromadb

# Now load the model:

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
# Step 2: Create the embedding function

def get_embedding(text: str):
    return model.encode(text).tolist()

# Test it:
get_embedding("What dataset was used in the paper?")[:3]

[-0.10325060784816742, 0.10225580632686615, -0.056256309151649475]

In [8]:
# Step 3: Create the vector store function

def create_vector_store():
    client = chromadb.Client()
    collection = client.get_or_create_collection(name="research_papers")
    return collection
# Because once you have embeddings, you need a place to store them and search them.

In [9]:
# Step 4: Create the function to add chunks
def add_chunks_to_collection(collection, chunks: list[str]):
    for i, chunk in enumerate(chunks):
        collection.add(
            documents=[chunk],
            embeddings=[get_embedding(chunk)],
            ids=[f"chunk_{i}"]
        )

In [10]:
# Step 5: Create the retrieval function
def retrieve_relevant_chunks(collection, question: str, top_k: int = 3):
    query_embedding = get_embedding(question)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    documents = results.get("documents", [[]])[0]
    ids = results.get("ids", [[]])[0]

    return list(zip(ids, documents))

In [11]:
# Step 6: Create some sample chunks
# Right now, to make sure retrieval works, use fake chunks first.

chunks = [
    "The paper uses the CIFAR-10 dataset for image classification experiments.",
    "The proposed model improves accuracy compared to baseline methods.",
    "Training was done for 50 epochs using the Adam optimizer.",
    "The main contribution of the paper is a lightweight transformer architecture."
]

In [12]:
# Step 7: Create the collection


collection = create_vector_store()

In [13]:
# Step 8: Add chunks into the collection
add_chunks_to_collection(collection, chunks)

# Now your vector store has data.
# Check it:
collection.count()

4

In [14]:
# Step 9: Ask a question and retrieve results

question = "What dataset was used in the paper?"
retrieved = retrieve_relevant_chunks(collection, question, top_k=3)
retrieved

[('chunk_0',
  'The paper uses the CIFAR-10 dataset for image classification experiments.'),
 ('chunk_2', 'Training was done for 50 epochs using the Adam optimizer.'),
 ('chunk_1',
  'The proposed model improves accuracy compared to baseline methods.')]

In [15]:
# Step 10: Print the results nicely

for chunk_id, doc in retrieved:
    print(f"{chunk_id}: {doc}\n")

chunk_0: The paper uses the CIFAR-10 dataset for image classification experiments.

chunk_2: Training was done for 50 epochs using the Adam optimizer.

chunk_1: The proposed model improves accuracy compared to baseline methods.

